# 프롬프트 엔지니어링 2 - 출력 형식 / 제약 / 컨텍스트 / Zero-shot·Few-shot (2026-03-25)

## 학습 목표
- 구조화된 **입출력 형식**(JSON, Markdown, YAML, XML)이 LLM 응답에 주는 영향 체감
- **ConstrainedPrompt** 클래스로 길이/내용/형식/스타일 제약을 체계적으로 주는 법
- **컨텍스트 제공**과 RAG의 관계, `Lost in the Middle` 논문 핵심
- **Zero-shot vs Few-shot** 차이와 활용 시나리오
- **카테고리 분류 / 톤 변환** 등 실제 응용

## 어제 복습 한 줄
> LLM은 "다음 토큰 확률 분포"로 문장을 생성 → 프롬프트는 그 분포를 **우리 의도에 맞게 쏠리게 만드는 장치**.

## 오늘의 비유
> LLM을 **AI 알바생**이라고 생각해보세요.
> - **어제**: 지시문 쓰는 법 + 역할 부여 + 단계별 지시
> - **오늘**: 결과물을 **어떤 양식지**에 써달라고 할지 + **하지 말아야 할 것 체크리스트** + **참고 자료 제공** + **샘플 답안 첨부**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w3_memory_prompt_engineering/llm_260325_prompt_engineering_2.ipynb)

## 0. Colab 환경 설정

In [ ]:
!pip install -q langchain langchain-community langchain-core langchain-openai openai

In [ ]:
# --- Colab 전용 ---
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

client = OpenAI()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

## 1. 구조화된 입출력 형식

### 왜 구조화?
- LLM의 답변을 **프로그램이 파싱해서 사용**하려면 일정한 형식이 필요
- **JSON / Markdown / YAML / XML** 등 계층 구조를 가진 포맷을 쓰면 LLM이 더 안정적으로 응답

### 3가지 스타일 비교
| 스타일 | 특징 |
|---|---|
| **Plain prompt** | 자연어만, 형식 지시 없음 → 응답 제각각 |
| **Markdown prompt** | `#`, `##`, `-` 같은 계층으로 지시 → LLM도 계층으로 응답 |
| **JSON prompt** | JSON 예시 스키마 직접 제시 → 거의 JSON으로 응답 |

> **비유**: 신입에게 "보고서 써줘" vs "이 양식지 채워줘" 차이.

In [ ]:
# 구조화된 입출력 형식: json, table (markdown), yaml, xml

In [ ]:
# [1] Plain prompt: 자연어만으로 지시
plain_prompt = """다음 제품 리뷰를 분석해줘. 전체 감정(긍정/부정/혼합), 1~5점 점수, 장점 목록, 단점 목록, 핵심 키워드 3개를 알려줘.
리뷰 : "이 노트북 정말 가벼워서 좋아요! 다만 키보드 타건감이 아쉽네요"
유효한 json만 출력하세요"""

print(llm.invoke([HumanMessage(content=plain_prompt)]).content)

In [ ]:
# [2] Markdown prompt: 헤더(#)와 불릿(-)으로 계층적 지시
# 웹의 많은 문서들이 Markdown/HTML 형태라 LLM은 이 구조에 익숙함
markdown_prompt = """# 제품 리뷰 감정 분석

## 입력 리뷰
> "이 노트북 정말 가벼워서 좋아요! 다만 키보드 타건감이 아쉽네요"

## 분석 항목
- **overall_sentiment** : 긍정/부정/혼합 중 하나
- **score** : 1~5점수
- **pros** : 장점 리스트
- **cons** : 단점 리스트
- **keywords** : 핵심 키워드 3개

## 출력 형식
유효한 json만 출력하세요. 다른 텍스트를 포함하지 마세요
"""

print(llm.invoke([HumanMessage(content=markdown_prompt)]).content)

In [ ]:
# [3] JSON prompt: 원하는 JSON 스키마를 직접 예시로 제공
json_prompt = """다음 제품 리뷰를 분석해서 json 형식으로 출력하세요

리뷰 : "이 노트북 정말 가벼워서 좋아요! 다만 키보드 타건감이 아쉽네요"

출력 형식
{
    "overall_sentiment" : 긍정/부정/혼합,
    "score" : 1~5점수,
    "pros" : [장점],
    "cons" : [단점],
    "keywords" : [키워드1, 키워드2, 키워드3]
}
"""

print(llm.invoke([HumanMessage(content=json_prompt)]).content)

### Output Parser vs 프롬프트 지시의 관계

- **Parser (JsonOutputParser 등)**: LLM이 이미 뱉은 답변을 **파싱**하는 역할. 답변이 완전한 JSON이 아니면 실패함.
- **프롬프트 지시 ("JSON으로 출력하세요")**: LLM이 JSON 비슷하게 내뱉도록 **유도**. 100% 보장은 안 됨.

### 실무 팁
> 두 개를 **같이** 써야 안정적!
> - 프롬프트로 JSON 형식 강하게 유도 + 예외처리와 함께 Parser 적용
> - HuggingFace 블로그의 **structured generation** 아티클도 참고

### 주의
논문 *"Let Me Speak Freely? A Study on the Impact of Format Restrictions on Performance of Large Language Models"* 에 따르면, **너무 강한 포맷 제약은 오히려 답변 품질을 떨어뜨릴 수 있음**. 적절한 균형이 필요.

## 2. 실습: 회의록 → JSON 구조화

자연어로 된 회의록을 JSON 스키마로 변환하는 함수 만들기.

In [ ]:
# Parser (출력 형식을 파싱하는 개념 리마인드)

In [ ]:
# 회의록 → 구조화된 데이터로 변환하는 함수
def generate_meeting_minute(raw_text, json_structure):
    """
    자연어 회의록을 받아 JSON 형태의 구조화된 회의록으로 변환.
    - raw_text: 회의 내용(자연어)
    - json_structure: 원하는 JSON 스키마(dict)
    """
    messages = [
        # System: 역할 부여 + 부연설명 금지 (순수 JSON만)
        SystemMessage(content='당신은 회의록 전문가입니다. 부연설명 없이 회의록을 json데이터만 반환하세요.'),
        # Human: 구조와 회의 내용 전달
        HumanMessage(content=f'다음 회의 내용을 지정된 구조로 정리해 주세요.\n {json_structure} \n\n회의내용: {raw_text}')
    ]
    return llm.invoke(messages).content

In [ ]:
# 원하는 JSON 스키마 정의 (딕셔너리 형태로 예시 제시)
json_structure = {
    'date': 'YYYY-MM-DD',
    'attendees': ['이름1', '이름2'],
    'agenda': ['안건1', '안건2'],
    'decisions': ['결정1'],
    'action_items': [{'assignee': '담당자', 'task': '작업', 'deadline': '기한'}]
}

# 자연어 회의록 예시
raw_text = """
3월 15일 마케팅팀 주간 회의. 참석: 김팀장, 이대리, 박사원. 신규 SNS 캠페인 예산 5000만원 확정. 이대리가 3월 22일까지 시안 준비. 박사원은 경쟁사 분석 보고서 3월 20일까지.
"""
# LLM에 회의록 변환 요청
response = generate_meeting_minute(raw_text, json_structure)
print(response)

## 3. 제약 조건 프롬프트 (Constrained Prompt)

### 제약의 4가지 카테고리
| 카테고리 | 예시 |
|---|---|
| **길이** | 3문장 이내, 100자 이내, 5개 불릿포인트 |
| **내용** | 검색된 정보로만 답변, 주어진 정보 안에서만, 추측 금지 |
| **형식** | JSON만 출력, 불릿포인트만 사용 |
| **스타일** | 친근한 톤, 전문가 톤, 2030 타겟 |

### 부정 제약 vs 긍정 제약
- ❌ `"거짓말로 지어내지 마세요"` (부정문)
- ✅ `"사실만 그대로 얘기하세요"` (긍정문)

**이유**: 토크나이징 관점에서 부정문은 구조가 복잡해짐 → 임베딩 벡터 만들 때 불리. 긍정문으로 표현하면 의미가 깔끔하게 전달됨.

> **비유**: "차 막히는 길로 가지 마"보다 "고속도로로 가"가 더 명확.

### 왜 클래스로 만들까?
- 매번 블로그 뒤져가며 프롬프트 짜지 말고, **팀/프로젝트의 표준 규약**으로 굳혀두면 일관성 확보

In [ ]:
# prompt: 제약조건
# 3문장으로 요약 (길이 제한)
# 검색된 정보로만 말하세요 (RAG, 정보 제한)
# Json 형태로만 출력 (포맷 제한)
# 부정 제약보다 긍정 제약이 더 정확한 결과를 도출
# 제약 규격 통일 → 클래스화

In [ ]:
class ConstrainedPrompt:
    """
    제약 조건을 체이닝으로 추가하는 프롬프트 빌더.
    메서드 체이닝 패턴 (`return self`) → 읽기 쉬운 플루언트 API.
    """
    def __init__(self, base_instruction):
        # 기본 지시 (예: "클라우드 장점 설명해주세요")
        self.instruction = base_instruction
        # 제약 조건들을 리스트로 누적
        self.constraints = []

    def add_length(self, description):
        # 길이 제약 추가 (예: "5개 불릿")
        self.constraints.append(f'[길이] {description}')
        return self  # 체이닝용

    def add_content(self, description):
        # 내용 제약 추가 (예: "비용 관점 반드시 포함")
        self.constraints.append(f'[내용] {description}')
        return self

    def add_format(self, description):
        # 형식 제약 추가 (예: "이모지로 시작")
        self.constraints.append(f'[포맷] {description}')
        return self

    def add_style(self, description):
        # 스타일 제약 추가 (예: "경영진 대상")
        self.constraints.append(f'[스타일] {description}')
        return self

    def build(self):
        # 최종 프롬프트 조립
        parts = [self.instruction, '\n제약조건:']
        for c in self.constraints:
            parts.append(f' - {c}')
        return '\n'.join(parts)

    def execute(self, **kwargs):
        """
        프롬프트 빌드 + LLM 호출까지 한번에.
        **kwargs: temperature, max_tokens 등 LLM 파라미터를 유연하게 받음
        (kwargs = 어떤 키워드 인자가 올지 미리 모를 때 쓰는 파이썬 문법)
        """
        llm = ChatOpenAI(
            model="gpt-4o-mini",
            api_key=api_key,
            temperature=kwargs.get('temperature', 0.7),
            max_tokens=kwargs.get('max_tokens', 1000)
        )
        prompt = self.build()
        return llm.invoke([HumanMessage(content=prompt)]).content

In [ ]:
# 체이닝 스타일로 제약 조건 누적 → 읽기 쉬운 API
result = (
    ConstrainedPrompt('클라우드 컴퓨팅의 장점을 설명해 주세요.')
      .add_length('다섯개의 불릿 포인트')
      .add_content('비용, 확장성, 보안 관점을 반드시 포함할 것')
      .add_format('각 포인트는 한 줄로, 이모지로 시작할 것')
      .add_style('IT 비전공 경영진을 대상으로 전문 용어에 괄호로 설명을 추가')
      .execute(temperature=0.3, max_tokens=1000)
)

In [ ]:
# 결과 확인: 모든 제약이 반영됐는지 체크
print(result)

### 실습: 광고 카피 + 상세 설명 동시 생성

In [ ]:
# ConstrainedPrompt를 이용해서 광고카피 / 상세설명 생성
product = "에어프로 맥스 무선 헤드폰"
features = ["40시간 배터리", "멀티포인트 연결", "30dB 노이즈캔슬링", "300g 경량 설계"]

# 한 번에 광고카피(짧고 감성적) + 상세설명(길고 신뢰감 있는) 둘 다 요구
response = (
    ConstrainedPrompt(f'{product} 광고카피/상세설명을 작성해 주세요. 특징: {features}')
      .add_length('광고카피는 20자 내외로 다섯개의 리스트로 출력해 줘. \n상세설명은 최대 두 문장, 200자 이내로 간결하게 작성해 줘.')
      .add_content('특징을 반드시 포함할 것')
      .add_format('슬로건 형태로 마침표 없이')
      .add_style('2030 타겟, 감성적이고 트렌디한 톤')
      .execute(temperature=1, max_tokens=500)
)
print(response)

## 4. 컨텍스트 제공 (Context Injection)

### 핵심 개념
LLM이 **모르거나 최신이 아닌 정보**를 물어보면 → 할루시네이션(환각) 발생.
→ 답에 필요한 **참고 문서를 프롬프트에 같이 넣어주면** 정확도↑

### RAG vs 단순 Context
| | RAG | 단순 Context 주입 |
|---|---|---|
| 출처 | Vector DB에서 검색된 청크 | 하드코딩/미리 준비된 문서 |
| 목적 | 실시간성/대용량 지식 보완 | 특정 상황 정보 제공 |
| 공통점 | 프롬프트에 문서를 함께 전달한다는 점 |

### 논문 - *Lost in the Middle* (Liu et al., 2023)
**핵심 발견**: 컨텍스트가 길어질수록 성능이 **U자 곡선**을 그림.
- 중요한 정보가 **맨 앞**에 있을 때 → 잘 활용
- **맨 끝**에 있을 때 → 잘 활용
- **중간**에 있을 때 → 성능 급락

**논문 링크**: https://arxiv.org/pdf/2307.03172

### 실무 교훈
- RAG의 청크 사이즈 튜닝 중요 (너무 작으면 자잘하고, 너무 크면 중간 정보 놓침)
- 중요한 정보는 컨텍스트의 **처음이나 끝**에 배치

> **비유**: 30페이지 보고서 받았을 때 사람도 **서론/결론만 정독**하고 중간은 대충 읽는 것과 비슷.

In [ ]:
# 컨텍스트 제공
# RAG 논문: 제공 문맥의 최앞단, 뒷단 내용을 중요하게 읽음
# https://arxiv.org/pdf/2307.03172

In [ ]:
# 참고 문서: 회사 재택근무 정책 (실제 시나리오 흉내)
company_policy = """
[모두컴퍼니 재택근무 정책 v2.3]
- 주 3일 재택, 2일 출근 (화/목 필수 출근)
- 재택근무 시 오전 9시까지 Slack 상태 '업무중' 설정 필수
- 해외 원격근무는 최대 연속 2주까지 가능 (사전 승인 필요)
- 야간근무(22시 이후) 시 익일 오후 출근 가능
- 재택근무 장비 지원금: 연 100만원 (영수증 제출)
"""

In [ ]:
# 문서에 없는 질문("한 달" vs 실제로는 "2주")을 물어서 할루시네이션 방지 테스트
question = '해외에서 한 달 동안 원격 근무를 할 수 있나요?'

# 프롬프트 설계: "문서에 없으면 명시되어 있지 않다"고 답하도록 명령
with_context = f'''
아래 회사 정책 문서를 참고하여 질문에 답하세요.
문서에 없는 내용은 "해당 정책 문서에 명시되어 있지 않습니다"라고 답하세요.

정책문서:
"""{company_policy}"""
질문: {question}
'''
print(llm.invoke([HumanMessage(content=with_context)]).content)

In [ ]:
def answer_with_context(context, question):
    """
    컨텍스트 기반 QA 함수.
    - 할루시네이션 방지 clause 포함
    - 답변 + 근거(CoT의 일종) 함께 요구 → 정확도 향상
    """
    # 주의사항 (clause)을 별도 변수로 관리
    clause = """
    중요: 반드시 제공된 문서의 내용만을 근거로 답하세요.
    문서에 없는 내용은 "해당 정책 문서에 명시되어 있지 않습니다"라고 답하세요.
    추측하거나 외부 지식을 사용하지 마세요.
    """

    # 답변 + 근거 인용 형식 지정 → CoT 효과
    prompt = f"""
    아래 참고 문서를 기반으로 질문에 답하세요.
    {clause}

    참고 문서:
    \'\'\'{context}\'\'\'

    질문: {question}

    답변 형식:
    - 답변: [핵심 답변]
    - 근거: [문서에서 관련된 부분 인용]
    """
    return llm.invoke([HumanMessage(content=prompt)]).content

In [ ]:
# 더 강한 질문으로 테스트 ("1년" - 말도 안되는 기간)
question = '해외에서 1년 동안 원격 근무를 할 수 있나요?'
result = answer_with_context(company_policy, question)
print(result)
# → 답변 형식 지정 + 근거 요구 덕분에 더 명확한 답변

## 5. (참고) Transformer 시각화

**Transformer** = 번역 구조에서 출발. `Encoder → Decoder`
- **Encoder**: 입력 단어들의 관계를 인코딩
- **Decoder**: 출력 토큰을 하나씩 생성 (GPT 계열은 Decoder-only)

### 직관적 시각화
https://poloclub.github.io/transformer-explainer/

> 이 사이트에서 실제로 토큰이 어떻게 흘러가는지 볼 수 있음.

In [ ]:
# transformer에 대한 직관적인 시각화 사이트
# https://poloclub.github.io/transformer-explainer/
# transformer: 번역 input(encoder) - output(decoder)
# encoding → 입력 단어들의 관계 (transformer 과정)
# decoding → 출력

## 6. Zero-shot vs Few-shot

### 용어 정리
`Shot` = **예시의 개수**

| 용어 | 의미 |
|---|---|
| **Zero-shot** | 예시 없이 바로 질문 |
| **One-shot** | 예시 1개 제공 후 질문 |
| **Few-shot** | 예시 3~5개 제공 후 질문 |

### 주의: 예전 ML/CV에서 쓰던 Shot과는 개념이 다름
- 옛 ML: "Zero-shot" = **학습 데이터가 없는 상태**에서 예측 (예: 축구공 사진 한 번도 안 본 모델이 축구공 찾기)
- LLM 세계: "Zero-shot" = **프롬프트에 예시가 없는 상태** (학습은 이미 된 상태)

### Few-shot의 장단점
- ✅ 출력 포맷을 정형화하기 쉬움 ("이런 식으로 답해줘")
- ❌ 예시 토큰이 많이 들어가서 **비용 증가**

> **비유**: Zero-shot = "이 문제 풀어줘", Few-shot = "이렇게 푸는 거야 (예시) → 이 문제 풀어줘". 과외 선생님이 예제 풀이 먼저 보여주고 문제 풀게 하는 것과 같음.

In [ ]:
# zero-shot, few-shot: 예시 제공 여부가 핵심
# 식대 지원금에 대한 내용이 포함되어 있지 않습니다 → 감정 분석해 주세요.
# 식대 지원금에 대한 내용이 포함되어 있지 않습니다:감정 → 제공된 문서에 해당하는 정보가 없습니다. 감정 분석해 주세요.

### 6.1 Zero-shot 실습: 뉴스 분류

In [ ]:
# zero-shot: 아무 예시 없이 response를 기대
categories = ["기술", "경제", "스포츠", "문화", "정치"]
news_articles = [
    "삼성전자가 차세대 AI 반도체 개발에 3조원을 투자한다고 발표했다.",
    "한국은행이 기준금리를 0.25%p 인하하며 경기 부양에 나섰다.",
    "손흥민이 프리미어리그 시즌 최다 도움을 기록하며 팀 승리를 이끌었다.",
    "국립현대미술관에서 한국 현대미술 50년 특별전이 개막했다."
]

In [ ]:
# Zero-shot 분류: 카테고리 목록만 주고 예시 없이 분류 요청
classify_prompt = f'''
  당신은 뉴스 분류 전문가입니다. 주어진 뉴스 기사를 다음 카테고리 중 하나로 분류하세요.
  반드시 아래 카테고리 이름만 출력하세요.

  카테고리: {', '.join(categories)}
'''
# 각 기사에 대해 반복
for article in news_articles:
    result = llm.invoke([
        SystemMessage(content=classify_prompt),
        HumanMessage(content=f'뉴스 기사: {article}')
    ]).content
    print(f' 기사: {article[:30]}...')
    print(f' 분류: {result}\n')

### 6.2 Few-shot 실습: 감정 분석 with JSON 출력

시스템 메시지로 역할 부여 + HumanMessage/AIMessage 페어로 예시 제공 → 마지막에 실제 쿼리.

In [ ]:
# few-shot: 예시로 출력 포맷을 정형화 (one-shot, few-shot)
# 단점: 토큰 비용이 많이 듬
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# 메시지 시퀀스 구성: System → (Human-AI 예시 페어 N개) → 실제 Human 질의
few_shot_messages = [
    # 시스템 메시지로 역할 부여
    SystemMessage(content='주어진 리뷰의 감정을 분석하세요'),

    # 예시 1: 순수 긍정
    HumanMessage(content="리뷰: '이 제품 정말 최고입니다! 강력 추천해요'"),
    AIMessage(content='{"sentiment" : "긍정", "score" : 0.95, "keywords" : ["최고", "강력 추천"]}'),

    # 예시 2: 순수 부정
    HumanMessage(content="리뷰: '배송도 느리고 제품 품질도 형편없네요!'"),
    AIMessage(content='{"sentiment" : "부정", "score" : 0.15, "keywords" : ["느리고", "형편없네요"]}'),

    # 예시 3: 혼합
    HumanMessage(content="리뷰: '가격 대비 그럭저럭 쓸만하네요'"),
    AIMessage(content='{"sentiment" : "혼합", "score" : 0.6, "keywords" : ["가격 대비", "쓸만함"]}'),

    # 실제 쿼리 (질의)
    HumanMessage(content="리뷰: '디자인은 예쁜데, 배터리가 빨리 닳아요. 전체적으로 보통입니다.'"),
]

print(llm.invoke(few_shot_messages).content)

### 6.3 Few-shot 실습: 톤 변환기 (informal → formal)

비격식 → 비즈니스 톤으로 변환하는 Few-shot 패턴.

In [ ]:
# few-shot 실습: examples처럼 test_message가 출력되도록 LLM에 요청
examples = [
    {"informal": "내일 미팅 좀 미룰 수 있을까? 갑자기 일이 생겼어.",
     "formal": "안녕하세요. 내일 예정된 미팅 일정 변경을 요청드립니다. 긴급한 업무가 발생하여 조율이 필요합니다. 가능한 대체 일정을 알려주시면 감사하겠습니다."},
    {"informal": "그 보고서 다 했어? 빨리 보내줘.",
     "formal": "안녕하세요. 요청드렸던 보고서 진행 상황을 확인드립니다. 완료되셨다면 전달 부탁드리며, 추가 시간이 필요하시면 말씀해 주세요."},
    {"informal": "이번 프로젝트 결과 별로인데 어떻게 할까?",
     "formal": "안녕하세요. 이번 프로젝트 결과에 대해 논의가 필요합니다. 개선 방안을 함께 검토하기 위해 미팅을 잡는 것이 어떨까요?"}
]

# 테스트 메시지 (새로운 비격식 표현)
test_messages = [
    "다음 주 워크숍 참석 못 할 것 같아. 다른 사람 보내도 돼?",
    "예산 좀 더 받을 수 있을까? 지금 부족해."
]

# 시스템 메시지로 역할 부여
messages = [
    SystemMessage(content="당신은 비즈니스 커뮤니케이션 전문가입니다. 비격식적인 문장을 격식있는 문장으로 바꾸어주세요.")
]

# 예시를 Human-AI 페어로 추가 → LLM이 패턴 학습
for example in examples:
    messages.append(HumanMessage(content=example["informal"]))
    messages.append(AIMessage(content=example["formal"]))

# 실제 테스트 메시지들을 마지막에 추가
for test_msg in test_messages:
    messages.append(HumanMessage(content=test_msg))

print(llm.invoke(messages).content)

## 7. 오늘의 정리

| 기법 | 핵심 |
|---|---|
| **구조화된 입출력** | Plain < Markdown < JSON (구체적일수록 파싱 쉬움) |
| **Parser + 프롬프트** | Parser만으로 포맷 100% 보장 X → 프롬프트 지시와 병용 |
| **ConstrainedPrompt** | 길이/내용/형식/스타일 제약을 체이닝으로 누적 |
| **긍정 제약 > 부정 제약** | 토크나이징/임베딩 측면에서 유리 |
| **컨텍스트 주입** | RAG와 유사. Lost in the Middle 논문 인지 |
| **Zero-shot** | 예시 없이 바로 지시 |
| **Few-shot** | Human-AI 페어로 예시 제공 → 출력 포맷 정형화 |

### 내일(03-26) 예고
- **프롬프트 엔지니어링 고급 + 평가(Evaluation)**
- LLM 응답 품질을 어떻게 측정할까?
- A/B 테스트, 자동 평가 기법

### 오늘의 한마디
> LLM은 **확률 분포**를 따라 토큰을 뽑는 기계.
> 프롬프트 엔지니어링은 그 확률 분포를 **내 의도 쪽으로 기울이는 기술**입니다.
> 역할, 단계, 형식, 제약, 컨텍스트, 예시 - 모두 같은 목표를 다른 각도에서 달성하는 도구.